# Lag effect Analysis

### "정책 효과가 강하게 나타난 시점일수록, 모델은 더 쉽게 결과(Y)를 예측할 수 있다"
1. 정책이 효과를 발휘하면, Y(조직성과/개인결과)가 X(정책포함 정보)에 더 강하게 종속된다.
- 정책 도입 전에는 조직성과(Y)가 다양한 외적 요인에 따라 들쭉날쭉 (예측 어려움).
- 정책이 효과를 발휘하면 → 특정 정책 조건(X)에 따라 Y가 더 일관되고 예측 가능하게 변화함.
- 즉, 정책이 Y에 설명력을 부여한다.

2. 모델은 설명력 높은 관계일수록 예측 성능이 높다.
- 머신러닝 모델의 본질: 입력 X와 출력 Y 사이의 규칙성을 찾아내는 것.
- 따라서, 만약 정책 효과로 인해 X (정책포함) → Y의 인과 경로가 강해졌다면,
- 모델은 더 쉽게 일반화 가능한 패턴을 학습함.
- 결과적으로, 정확도, F1, AUC 등의 성능이 높아짐.

1. Stage 1
    - 2020년 데이터 X + 2021년 데이터 y => 모델-2020_2021
    - 2020년 데이터 X + 2023년 데이터 y => 모델-2020_2023
    => 각 모델의 퍼포먼스 측정 (강한 예측력을 보이는 모델 = 그게 결국 영향력)
    => 각 모델 해석력 지수 얻기 / 중요도에 따라서 인풋값의 가중치 다르게 줌


2. Stage 2
    - 2021년 데이터 X + 2021년 데이터 y => 모델-2021_2021
    - 2021년 데이터 X + 2023년 데이터 y => 모델-2022_2023

3. Stage 3
    - 2022년 데이터 X + 2023년 데이터 y => 모델-2022_2023

4. Stage 4
    - 2023년 데이터 X + 2023년 데이터 y => 모델-2023_2023

---

- Dataset: 2020 / 2021 / 2022 / 2023, 근데 2021 이랑 2023 데이터만 target label 갖고 있음
- Y : Target label
    - ~~2021 : C21C05_01H1 (8) 전체 숙련수준  / C21C05_01H2 (8) 전체 경쟁력~~
    - ~~2023 : C23C05_01H1 (8) 전체 숙련수준  / C23C05_01H2 (8) 전체 경쟁력~~ => 이렇게 하려했는데 C21C05_01H1의 Nan값이 88%
    - 2021 : C21C05_01H2 (8) 전체 경쟁력
    - 2023 : C23C05_01H2 (8) 전체 경쟁력

- X :

In [10]:
import matplotlib.pyplot as plt
import utils_, config
#from github.V1.utils_ import data_processing
import os
import numpy as np
import pandas as pd
from collections import defaultdict



def get_all_data(year_list):
    file_names = config.file_names       # Expected file names (e.g., ['file1.csv', 'file2.csv'])
    file_list = os.listdir(config.path)  # All files in the directory
    dataset = {}                         # Final dictionary to store data
    year_cnt = -1                        # Counter to map years to files

    for expected_file in file_names:
        for actual_file in file_list:
            if expected_file == actual_file:
                year_cnt += 1
                current_year = year_list[year_cnt]
                print(f"{expected_file} ===> {current_year} data")

                # Load data
                df, meta = utils_.data_import(os.path.join(config.path, expected_file))

                # Store in dataset dict with year as key
                dataset[current_year] = {
                    'data': df,
                    'meta': meta
                }

    return dataset


def analyze_dataframe_step1(df: pd.DataFrame, meta):
    non_numeric_info = {}
    Meta_col = []
    total_rows = len(df)

    for idx, col in enumerate(df.columns):
        # 숫자 변환 불가능한 값 마스크
        non_numeric_mask = ~pd.to_numeric(df[col], errors='coerce').notna()
        non_numeric_count = non_numeric_mask.sum()
        nan_count = df[col].isna().sum()

        if non_numeric_count > 0:
            non_numeric_info[col] = {
                'non_numeric': non_numeric_count,
                'nan': nan_count
            }
            Meta_col.append(idx)

    # 결과 출력
    print("\n\n🔍 숫자가 아닌 character가 포함된 컬럼들 및 해당 row 수:")
    for idx, (col, stats) in enumerate(non_numeric_info.items()):
        label = meta.column_labels[Meta_col[idx]]
        print(f"- {col} ({label}): 숫자 아님 {stats['non_numeric']}개 / NaN {stats['nan']}개")



def clean_dataframe_step1(df: pd.DataFrame, columns_to_drop: list):
    print("\n=================================\nPreprocessing\n=================================\n")
    total_rows = len(df)

    for idx, col in enumerate(df.columns):
        nan_count = df[col].isna().sum()

        # NaN이 25% 미만이면 평균으로 대체
        if nan_count > 0 and (nan_count / total_rows) < 0.25:
            try:
                mean_val = pd.to_numeric(df[col], errors='coerce').mean()
                df[col] = df[col].fillna(int(mean_val))
                print(f"→ {col}: NaN {nan_count}개 평균({int(mean_val):.2f})으로 대체 완료")
            except:
                print("평균 계산 불가 (비숫자형 포함 등)")
                pass  # 평균 계산 불가 (비숫자형 포함 등)

    # 지정된 컬럼 삭제
    cleaned_df = df.drop(columns=columns_to_drop, errors='ignore')

    return cleaned_df


def target_variable_check(dataset):
    for year in ['2021', '2023']:
        df, meta = dataset[year]['data'], dataset[year]['meta']
        col_names = [f'C{year[2:]}C05_01H1', f'C{year[2:]}C05_01H2']
        for col_name in col_names:
            nan_count = df[col_name].isna().sum()
            total = df.shape[0]
            print(f"{year} - {col_name} : NaN {nan_count}개 / 전체 {total}개 ({nan_count / total:.2%})")


In [11]:
year_list = config.year               # List of years (e.g., [2018, 2020, 2022])

dataset = get_all_data(year_list)

HCCP_2ndWave_Head_1st(최종).sav ===> 2020 data
HCCP_2ndWave_Head_2nd(최종).sav ===> 2021 data
HCCP_2ndWave_Head_3rd(최종).sav ===> 2022 data
HCCP_2ndWave_Head_4th.sav ===> 2023 data


In [14]:
#year_list = [2020]

for year in ['2021']:
    #print(dataset[year]['data'].shape)
    df, meta = dataset[year]['data'], dataset[year]['meta']
    analyze_dataframe_step1(df, meta)
    new_df = clean_dataframe_step1(df, columns_to_drop=[f'C{year[2:]}_ID1'])  #기업아이디 삭제



🔍 숫자가 아닌 character가 포함된 컬럼들 및 해당 row 수:
- C21_SEX2 (대표자성별 2): 숫자 아님 387개 / NaN 387개
- C21_SEX3 (대표자성별 3): 숫자 아님 490개 / NaN 490개
- C21A01_021 (외국인 지분 역할): 숫자 아님 399개 / NaN 399개
- C21A01_061 (신제품의 개발자): 숫자 아님 325개 / NaN 325개
- C21A01_062A (신제품의 시장출시로 인한 효과-1순위): 숫자 아님 325개 / NaN 325개
- C21A01_062B (신제품의 시장출시로 인한 효과-2순위): 숫자 아님 328개 / NaN 328개
- C21A01_07B (신제품 또는 개선제품 출시를 방해한 요인-2순위): 숫자 아님 232개 / NaN 232개
- C21A02_08A (해외 법인 기능_생산): 숫자 아님 376개 / NaN 376개
- C21A02_08B (해외 법인 기능_판매): 숫자 아님 382개 / NaN 382개
- C21A02_08C (해외 법인 기능_개발): 숫자 아님 473개 / NaN 473개
- C21A02_08D (해외 법인 기능_물류): 숫자 아님 462개 / NaN 462개
- C21A02_08E (해외 법인 기능_기타): 숫자 아님 498개 / NaN 498개
- C21A02_08F (해외 법인 기능_해외법인 없음): 숫자 아님 181개 / NaN 181개
- C21B02_05F3 ((6)생산기능직 정규직 여자 인원): 숫자 아님 1개 / NaN 1개
- C21B02_06B (정규직 전환 인원): 숫자 아님 357개 / NaN 357개
- C21B02_08B (직접고용 전환 인원): 숫자 아님 444개 / NaN 444개
- C21B02_10A1 ((1)남자-사원급 인원(인력현황)): 숫자 아님 20개 / NaN 20개
- C21B02_10A2 ((1)남자-대리급 인원(인력현황)): 숫자 아님 20개 / NaN 20개
- C21B02_10A3 ((1)남자-과

In [28]:
#utils_.see_col_idx_and_name(dataset[year]['data'], dataset[year]['meta'])
#df['C23C05_01H1'].value_counts()
target_variable_check(dataset)

2021 - C21C05_01H1 : NaN 443개 / 전체 500개 (88.60%)
2021 - C21C05_01H2 : NaN 0개 / 전체 500개 (0.00%)
2023 - C23C05_01H1 : NaN 0개 / 전체 500개 (0.00%)
2023 - C23C05_01H2 : NaN 0개 / 전체 500개 (0.00%)


In [20]:
#new_df['C21C05_01H1'].value_counts()
df['C21C05_01H1'].value_counts()

C21C05_01H1
3.0    38
4.0    12
5.0     3
2.0     2
1.0     2
Name: count, dtype: int64

In [17]:
new_df['C21C05_01H2'].value_counts()

C21C05_01H2
3.0    386
4.0     72
2.0     35
1.0      4
5.0      3
Name: count, dtype: int64

In [1]:
import utils_, config, model

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

def main():
    for i in config.Target_Indices:

        aadf, meta = utils_.data_import(config.file_path)
        print(f"\n==============================================\n CLASS: {meta.column_labels[i]}\n==============================================\n\n")
        X, Y = utils_.data_processing(df, target_idx=i)
        #X = HRD_col_select(all_X)
        X_train, X_test, y_train, y_test= utils_.data_aug_smote(X, Y)
        col_name = df.columns[i]

        GB_model = model.GB(X_train, X_test, y_train, y_test)
        XGB_model = model.XGBoost(X_train, X_test, y_train, y_test, col_name)
        #LGBM_model = models_compare_all.LightGBM(X_train, X_test, y_train, y_test)
        #CatBoost_model = models_compare_all.CatBoost(X_train, X_test, y_train, y_test)
        #RF_model = models_compare_all.RF(X_train, X_test, y_train, y_test)
        #DT = models_compare_all.Dec_T(X_train, X_test, y_train, y_test)

        model.feature_importance(XGB_model, X, df, meta)


    return None

In [ ]:
if __name__ == "__main__":
    main()